# SF9 Hypothesis Tests: H1 / H2 / H3

Reproduces the CC-006a SF9 refinements from the companion paper (Nechepurenko 2026, Paper 1).

These results are **NOT** in the headline dataset (`markets-stylized-facts-v1.parquet`).  
They are provided here for transparency and reproducibility.

**Source data:** `evaluation/output/sf9_refined.json` from the companion repository.  
**Dependencies:** `pandas`, `matplotlib`

## Summary of findings
- **H1 (per-region depth):** Depth growth (12h-3h → 3h-1h) is strongest in `mid` and `high` index regions; `boundary-low` and `boundary-high` markets show flatter profiles
- **H2 (near-mid 50bps depth):** Pooled median depth within 50bps of mid is **structurally zero** in all buckets except `1h-5m`. This is Empirical Condition 1.
- **H3 (by-path cohort):** Depth accumulation pattern differs between `decided-early`, `contested`, and `middle` cohorts

In [ ]:
import json
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Adjust path to sf9_refined.json relative to your project root
SF9_PATH = pathlib.Path("../../../../evaluation/output/sf9_refined.json")

with open(SF9_PATH) as f:
    sf9r = json.load(f)

BUCKETS = ["24h-12h", "12h-3h", "3h-1h", "1h-5m", "5m-0"]
REGIONS = ["boundary-low", "low", "mid", "high", "boundary-high"]
print("SF9 refined data loaded.")
print("Keys:", list(sf9r.keys()))

## H1 — Depth by index region

For each of 5 index regions, compute depth medians across the 5 time-to-resolution buckets and the contraction factors between adjacent buckets.

In [ ]:
h1 = sf9r["sf9_H1_conditional"]

rows = []
for region in REGIONS:
    rdata = h1.get(region, {})
    medians = rdata.get("bucket_medians", {})
    n_mkts  = rdata.get("n_markets_per_bucket", {})
    cf      = rdata.get("contraction_factors", [])
    for i, bucket in enumerate(BUCKETS):
        rows.append({
            "region": region,
            "bucket": bucket,
            "median_depth_usdc": medians.get(bucket),
            "n_markets": n_mkts.get(bucket, 0),
        })

df_h1 = pd.DataFrame(rows)
pivot_h1 = df_h1.pivot(index="region", columns="bucket", values="median_depth_usdc")
pivot_h1 = pivot_h1[BUCKETS]  # enforce bucket order

print("H1 — Pooled median depth (USDC) by index region and time-to-resolution bucket:")
print(pivot_h1.to_string(float_format="{:,.0f}".format))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for region in REGIONS:
    vals = [df_h1[(df_h1.region==region) & (df_h1.bucket==b)]["median_depth_usdc"].values
            for b in BUCKETS]
    vals = [v[0] if len(v) > 0 and v[0] is not None else float("nan") for v in vals]
    ax.plot(BUCKETS, vals, marker="o", label=region)

ax.set_title("H1: Depth by index region and time-to-resolution bucket (200bps window)")
ax.set_xlabel("Time-to-resolution bucket")
ax.set_ylabel("Pooled median depth (USDC)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend()
fig.tight_layout()
plt.show()

## H2 — Near-mid depth (50bps window)

Empirical Condition 1: median depth within 50bps of mid is structurally zero throughout the market lifecycle. Only the `1h-5m` bucket shows nonzero median depth.

In [ ]:
h2 = sf9r["sf9_H2_near_mid"]
classes = ["sports", "other", "crypto", "politics", "pooled"]

rows_h2 = []
for cls in classes:
    cdata = h2.get(cls, {})
    medians = cdata.get("bucket_medians", {})
    n_mkts  = cdata.get("n_markets_per_bucket", {})
    for bucket in BUCKETS:
        rows_h2.append({
            "class": cls,
            "bucket": bucket,
            "median_depth_50bps_usdc": medians.get(bucket),
            "n_markets": n_mkts.get(bucket, 0),
        })

df_h2 = pd.DataFrame(rows_h2)
pivot_h2 = df_h2.pivot(index="class", columns="bucket", values="median_depth_50bps_usdc")
pivot_h2 = pivot_h2[BUCKETS]

print("H2 — Median depth within 50bps of mid (USDC) by class and bucket:")
print(pivot_h2.to_string(float_format="{:,.1f}".format))
print()
print("Interpretation: Structural zeros throughout all buckets confirm Empirical Condition 1.")
print("The 1h-5m outlier (pooled median 219 USDC) reflects a handful of high-liquidity crypto markets.")

## H3 — By-path cohort

Markets split into three cohorts based on their price trajectory:
- `decided-early`: index reached >90% or <10% within the first 20% of market lifetime
- `contested`: index remained in [40%, 60%] for majority of lifetime  
- `middle`: neither of the above

In [ ]:
h3 = sf9r["sf9_H3_by_path"]
cohorts = ["decided-early", "contested", "middle"]

rows_h3 = []
for cohort in cohorts:
    cdata = h3.get(cohort, {})
    medians = cdata.get("bucket_medians", {}) if isinstance(cdata, dict) else {}
    n_mkts  = cdata.get("n_markets_per_bucket", {}) if isinstance(cdata, dict) else {}
    cf      = cdata.get("contraction_factors", []) if isinstance(cdata, dict) else []
    for bucket in BUCKETS:
        rows_h3.append({
            "cohort": cohort,
            "bucket": bucket,
            "median_depth_usdc": medians.get(bucket),
            "n_markets": n_mkts.get(bucket, 0),
        })

df_h3 = pd.DataFrame(rows_h3)
pivot_h3 = df_h3.pivot(index="cohort", columns="bucket", values="median_depth_usdc")
pivot_h3 = pivot_h3[BUCKETS]

print("H3 — Median depth (200bps window, USDC) by path cohort and bucket:")
print(pivot_h3.to_string(float_format="{:,.0f}".format))
print()

summary = h3.get("_path_assignment_summary", {})
n_with_path = h3.get("_n_markets_with_path_data", "?")
print(f"Markets with path data: {n_with_path}")
print("Path assignment summary:", summary)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cohort in cohorts:
    vals = [df_h3[(df_h3.cohort==cohort) & (df_h3.bucket==b)]["median_depth_usdc"].values
            for b in BUCKETS]
    vals = [v[0] if len(v) > 0 and v[0] is not None else float("nan") for v in vals]
    ax.plot(BUCKETS, vals, marker="o", label=cohort)

ax.set_title("H3: Depth by path cohort and time-to-resolution bucket (200bps window)")
ax.set_xlabel("Time-to-resolution bucket")
ax.set_ylabel("Pooled median depth (USDC)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend()
fig.tight_layout()
plt.show()

---

## Summary

| Hypothesis | Finding | In headline dataset? |
|---|---|---|
| H1 (per-region depth) | `mid` and `high` regions drive the 5x growth at 200bps | No — aggregate only |
| H2 (near-mid 50bps) | Structural zeros throughout → Empirical Condition 1 | No — aggregate only |
| H3 (by-path cohort) | `decided-early` markets show earlier depth accumulation | No — aggregate only |

These refinements inform the theoretical framing in Paper 1 §5.6 but are not part of the headline dataset to keep the schema focused on per-market measurements with broad coverage.

Source: `evaluation/output/sf9_refined.json` (CC-006a, companion repository).